In [1]:
# EDA showed visual differences.
# Statistical testing now measures whether those differences
# have both statistical evidence and meaningful magnitude.

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import (
    mannwhitneyu,
    chi2_contingency,
)

from statsmodels.stats.multitest import (
    multipletests,
)


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


df = pd.read_parquet(
    PROCESSED_DIR
    / "customer_feature_sample.parquet"
)


print(
    f"Customers loaded: {len(df):,}"
)

Customers loaded: 458,913


In [2]:
CANDIDATE_FEATURES = [
    "P_2",
    "B_1",
    "B_2",
    "B_9",
    "D_39",
    "D_41",
    "D_44",
    "D_45",
    "D_47",
    "R_1",
    "R_2",
    "R_3",
    "S_3",
]


candidate_features = [
    feature
    for feature
    in CANDIDATE_FEATURES
    if feature in df.columns
]


candidate_features

['P_2',
 'B_1',
 'B_2',
 'B_9',
 'D_39',
 'D_41',
 'D_44',
 'D_45',
 'D_47',
 'R_1',
 'R_2',
 'R_3',
 'S_3']

In [3]:
# Mann-Whitney does not require normally distributed features.
# Rank-biserial correlation gives us an interpretable effect size.

def rank_biserial(
    u,
    n1,
    n2,
):

    return (
        2 * u
        / (n1 * n2)
        - 1
    )


def feature_family(
    feature,
):

    return feature.split(
        "_"
    )[0]

In [4]:
results = []


for feature in candidate_features:

    default = (
        df.loc[
            df[
                "target"
            ] == 1,
            feature,
        ]
        .dropna()
    )

    non_default = (
        df.loc[
            df[
                "target"
            ] == 0,
            feature,
        ]
        .dropna()
    )

    u, p = mannwhitneyu(
        default,
        non_default,
        alternative="two-sided",
    )

    effect = rank_biserial(
        u,
        len(default),
        len(non_default),
    )

    results.append(
        {
            "feature":
                feature,

            "family":
                feature_family(
                    feature
                ),

            "default_median":
                default.median(),

            "non_default_median":
                non_default.median(),

            "u_statistic":
                u,

            "p_value":
                p,

            "effect_size":
                effect,

            "abs_effect_size":
                abs(effect),

            "missing_pct":
                df[
                    feature
                ]
                .isna()
                .mean()
                * 100,
        }
    )


results = pd.DataFrame(
    results
)


results

,feature,family,default_median,non_default_median,u_statistic,p_value,effect_size,abs_effect_size,missing_pct
0,P_2,P,0.351809,0.784433,3.323217e+09,0.0,-0.833633,0.833633,0.646964
1,B_1,B,0.207850,0.019613,3.375996e+10,0.0,0.670805,0.670805,0.000000
2,B_2,B,0.041610,0.817258,6.487561e+09,0.0,-0.678884,0.678884,0.006755
3,B_9,B,0.586549,0.009342,3.342211e+10,0.0,0.654084,0.654084,0.000000
4,D_39,D,0.093782,0.008301,2.740551e+10,0.0,0.356318,0.356318,0.000000
5,D_41,D,0.008422,0.005464,2.706265e+10,0.0,0.339523,0.339523,0.006755
6,D_44,D,0.257234,0.006370,3.095335e+10,0.0,0.688698,0.688698,4.858219
7,D_45,D,0.061716,0.219505,1.186843e+10,0.0,-0.412547,0.412547,0.006755
8,D_47,D,0.249048,0.418526,1.207259e+10,0.0,-0.402519,0.402519,0.000000
9,R_1,R,0.009184,0.005282,2.849337e+10,0.0,0.410157,0.410157,0.000000


In [5]:
# With many tests, some small p-values can occur by chance.
# FDR correction keeps feature selection more defensible.

results[
    "p_value_fdr"
] = multipletests(
    results[
        "p_value"
    ],
    method="fdr_bh",
)[1]


results = (
    results
    .sort_values(
        "abs_effect_size",
        ascending=False,
    )
)


results

,feature,family,default_median,non_default_median,u_statistic,p_value,effect_size,abs_effect_size,missing_pct,p_value_fdr
0,P_2,P,0.351809,0.784433,3.323217e+09,0.0,-0.833633,0.833633,0.646964,0.0
6,D_44,D,0.257234,0.006370,3.095335e+10,0.0,0.688698,0.688698,4.858219,0.0
2,B_2,B,0.041610,0.817258,6.487561e+09,0.0,-0.678884,0.678884,0.006755,0.0
1,B_1,B,0.207850,0.019613,3.375996e+10,0.0,0.670805,0.670805,0.000000,0.0
3,B_9,B,0.586549,0.009342,3.342211e+10,0.0,0.654084,0.654084,0.000000,0.0
12,S_3,S,0.248055,0.151809,2.192973e+10,0.0,0.512695,0.512695,18.515492,0.0
7,D_45,D,0.061716,0.219505,1.186843e+10,0.0,-0.412547,0.412547,0.006755,0.0
9,R_1,R,0.009184,0.005282,2.849337e+10,0.0,0.410157,0.410157,0.000000,0.0
8,D_47,D,0.249048,0.418526,1.207259e+10,0.0,-0.402519,0.402519,0.000000,0.0
4,D_39,D,0.093782,0.008301,2.740551e+10,0.0,0.356318,0.356318,0.000000,0.0


In [6]:
# This ranks whole feature families instead of only individual variables.

family_summary = (
    results
    .groupby(
        "family",
        as_index=False,
    )
    .agg(
        features_tested=(
            "feature",
            "count",
        ),
        median_effect=(
            "abs_effect_size",
            "median",
        ),
        strongest_effect=(
            "abs_effect_size",
            "max",
        ),
        significant_features=(
            "p_value_fdr",
            lambda x:
                (x < 0.05).sum(),
        ),
    )
    .sort_values(
        "median_effect",
        ascending=False,
    )
)


family_summary

,family,features_tested,median_effect,strongest_effect,significant_features
2,P,1,0.833633,0.833633,1
0,B,3,0.670805,0.678884,3
4,S,1,0.512695,0.512695,1
1,D,5,0.402519,0.688698,5
3,R,3,0.306963,0.410157,3


In [7]:
# Categorical features are tested with chi-square / Cramér's V.
# They come from the expanded latest-statement sample, not from the
# numeric-only parquet used above.

df_cat = pd.read_parquet(
    PROCESSED_DIR
    / "customer_feature_sample_with_categoricals.parquet"
)


CATEGORICAL_FEATURES = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
]


def cramers_v(
    table,
):

    chi2 = (
        chi2_contingency(
            table
        )[0]
    )

    n = (
        table
        .to_numpy()
        .sum()
    )

    rows, cols = (
        table.shape
    )

    denominator = (
        n
        * min(
            rows - 1,
            cols - 1,
        )
    )

    if denominator == 0:
        return np.nan

    return np.sqrt(
        chi2
        / denominator
    )


categorical_results = []


for feature in CATEGORICAL_FEATURES:

    if feature not in df_cat.columns:
        continue

    temp = df_cat[
        [
            feature,
            "target",
        ]
    ].dropna()

    table = pd.crosstab(
        temp[
            feature
        ],
        temp[
            "target"
        ],
    )

    if (
        table.shape[0] < 2
        or table.shape[1] < 2
    ):
        continue

    chi2, p, dof, expected = (
        chi2_contingency(
            table
        )
    )

    categorical_results.append(
        {
            "feature":
                feature,

            "family":
                feature_family(
                    feature
                ),

            "test_type":
                "chi_square",

            "chi_square":
                chi2,

            "p_value":
                p,

            "effect_size":
                cramers_v(
                    table
                ),

            "abs_effect_size":
                cramers_v(
                    table
                ),

            "missing_pct":
                df_cat[
                    feature
                ]
                .isna()
                .mean()
                * 100,

            "n_levels":
                table.shape[0],
        }
    )


categorical_results = pd.DataFrame(
    categorical_results
)


print(
    "Categorical features tested:",
    len(categorical_results),
)


categorical_results


Categorical features tested: 10


,feature,family,test_type,chi_square,p_value,effect_size,abs_effect_size,missing_pct,n_levels
0,B_30,B,chi_square,75874.675273,0.000000e+00,0.406629,0.406629,0.006755,3
1,B_38,B,chi_square,128083.127597,0.000000e+00,0.528318,0.528318,0.006755,7
2,D_114,D,chi_square,16892.166846,0.000000e+00,0.193127,0.193127,1.311142,2
3,D_116,D,chi_square,1402.394512,6.340111e-307,0.055646,0.055646,1.311142,2
4,D_117,D,chi_square,14250.017932,0.000000e+00,0.177382,0.177382,1.311142,7
5,D_120,D,chi_square,26374.778427,0.000000e+00,0.241321,0.241321,1.311142,2
6,D_126,D,chi_square,1391.944128,1.183016e-304,0.055074,0.055074,0.000000,2
7,D_63,D,chi_square,3636.386434,0.000000e+00,0.089016,0.089016,0.000000,6
8,D_64,D,chi_square,18042.047160,0.000000e+00,0.200484,0.200484,2.187125,3
9,D_68,D,chi_square,17339.713331,0.000000e+00,0.196319,0.196319,1.963771,6


In [8]:
# FDR is recomputed once across numeric and categorical tests.
# This replaces the numeric-only correction from the earlier cell.

results = results.copy()
results["test_type"] = "mannwhitney"

combined_tests = pd.concat(
    [
        results[
            [
                "feature",
                "p_value",
                "test_type",
            ]
        ],
        categorical_results[
            [
                "feature",
                "p_value",
                "test_type",
            ]
        ],
    ],
    ignore_index=True,
)


combined_tests[
    "p_value_fdr"
] = multipletests(
    combined_tests[
        "p_value"
    ],
    method="fdr_bh",
)[1]


fdr_map = combined_tests.set_index(
    "feature"
)[
    "p_value_fdr"
]


results[
    "p_value_fdr"
] = results[
    "feature"
].map(
    fdr_map
)


categorical_results[
    "p_value_fdr"
] = categorical_results[
    "feature"
].map(
    fdr_map
)


print(
    "Joint tests:",
    len(combined_tests),
)


categorical_results.sort_values(
    "abs_effect_size",
    ascending=False,
)


Joint tests: 23


,feature,family,test_type,chi_square,p_value,effect_size,abs_effect_size,missing_pct,n_levels,p_value_fdr
1,B_38,B,chi_square,128083.127597,0.000000e+00,0.528318,0.528318,0.006755,7,0.000000e+00
0,B_30,B,chi_square,75874.675273,0.000000e+00,0.406629,0.406629,0.006755,3,0.000000e+00
5,D_120,D,chi_square,26374.778427,0.000000e+00,0.241321,0.241321,1.311142,2,0.000000e+00
8,D_64,D,chi_square,18042.047160,0.000000e+00,0.200484,0.200484,2.187125,3,0.000000e+00
9,D_68,D,chi_square,17339.713331,0.000000e+00,0.196319,0.196319,1.963771,6,0.000000e+00
2,D_114,D,chi_square,16892.166846,0.000000e+00,0.193127,0.193127,1.311142,2,0.000000e+00
4,D_117,D,chi_square,14250.017932,0.000000e+00,0.177382,0.177382,1.311142,7,0.000000e+00
7,D_63,D,chi_square,3636.386434,0.000000e+00,0.089016,0.089016,0.000000,6,0.000000e+00
3,D_116,D,chi_square,1402.394512,6.340111e-307,0.055646,0.055646,1.311142,2,6.628298e-307
6,D_126,D,chi_square,1391.944128,1.183016e-304,0.055074,0.055074,0.000000,2,1.183016e-304


In [9]:
# Statistical significance alone is not enough in such a large dataset.
# We require corrected significance, non-trivial effect size,
# and reasonable missingness.
# Numeric effect size is |rank-biserial|; categorical is Cramér's V.
# The same 0.10 threshold is used for both.

numeric_shortlist = results.loc[
    (
        results[
            "p_value_fdr"
        ] < 0.05
    )
    &
    (
        results[
            "abs_effect_size"
        ] >= 0.10
    )
    &
    (
        results[
            "missing_pct"
        ] < 60
    )
].copy()


categorical_shortlist = categorical_results.loc[
    (
        categorical_results[
            "p_value_fdr"
        ] < 0.05
    )
    &
    (
        categorical_results[
            "abs_effect_size"
        ] >= 0.10
    )
    &
    (
        categorical_results[
            "missing_pct"
        ] < 60
    )
].copy()


shortlist = pd.concat(
    [
        numeric_shortlist,
        categorical_shortlist,
    ],
    ignore_index=True,
    sort=False,
)


if len(shortlist) < 5:

    shortlist = (
        pd.concat(
            [
                results.head(10),
                categorical_results.head(5),
            ],
            ignore_index=True,
            sort=False,
        )
    )


shortlist = (
    shortlist
    .sort_values(
        "abs_effect_size",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "Numeric shortlist:",
    len(numeric_shortlist),
)

print(
    "Categorical shortlist:",
    len(categorical_shortlist),
)


shortlist


Numeric shortlist: 13
Categorical shortlist: 7


,feature,family,default_median,non_default_median,u_statistic,p_value,effect_size,abs_effect_size,missing_pct,p_value_fdr,test_type,chi_square,n_levels
0,P_2,P,0.351809,0.784433,3.323217e+09,0.0,-0.833633,0.833633,0.646964,0.0,mannwhitney,NaN,NaN
1,D_44,D,0.257234,0.006370,3.095335e+10,0.0,0.688698,0.688698,4.858219,0.0,mannwhitney,NaN,NaN
2,B_2,B,0.041610,0.817258,6.487561e+09,0.0,-0.678884,0.678884,0.006755,0.0,mannwhitney,NaN,NaN
3,B_1,B,0.207850,0.019613,3.375996e+10,0.0,0.670805,0.670805,0.000000,0.0,mannwhitney,NaN,NaN
4,B_9,B,0.586549,0.009342,3.342211e+10,0.0,0.654084,0.654084,0.000000,0.0,mannwhitney,NaN,NaN
5,B_38,B,NaN,NaN,NaN,0.0,0.528318,0.528318,0.006755,0.0,chi_square,128083.127597,7.0
6,S_3,S,0.248055,0.151809,2.192973e+10,0.0,0.512695,0.512695,18.515492,0.0,mannwhitney,NaN,NaN
7,D_45,D,0.061716,0.219505,1.186843e+10,0.0,-0.412547,0.412547,0.006755,0.0,mannwhitney,NaN,NaN
8,R_1,R,0.009184,0.005282,2.849337e+10,0.0,0.410157,0.410157,0.000000,0.0,mannwhitney,NaN,NaN
9,B_30,B,NaN,NaN,NaN,0.0,0.406629,0.406629,0.006755,0.0,chi_square,75874.675273,3.0


In [10]:
results.to_csv(
    PROCESSED_DIR
    / "statistical_test_results.csv",
    index=False,
)


categorical_results.to_csv(
    PROCESSED_DIR
    / "statistical_categorical_results.csv",
    index=False,
)


family_summary.to_csv(
    PROCESSED_DIR
    / "statistical_family_summary.csv",
    index=False,
)


shortlist.to_csv(
    PROCESSED_DIR
    / "statistical_shortlist.csv",
    index=False,
)


print(
    "Statistical testing outputs saved."
)

print(
    "Shortlist features:",
    shortlist[
        "feature"
    ].tolist()
)


Statistical testing outputs saved.
Shortlist features: ['P_2', 'D_44', 'B_2', 'B_1', 'B_9', 'B_38', 'S_3', 'D_45', 'R_1', 'B_30', 'D_47', 'D_39', 'D_41', 'R_3', 'R_2', 'D_120', 'D_64', 'D_68', 'D_114', 'D_117']


## Summary

This notebook tests whether latest-statement features separate default from non-default customers.

**What we did**
- Loaded `customer_feature_sample.parquet` for 13 numeric features.
- Ran Mann–Whitney U tests and rank-biserial effect sizes.
- Loaded `customer_feature_sample_with_categoricals.parquet` for 11 known AMEX categoricals.
- Ran chi-square tests with Cramér's V. `D_66` had only one observed level after dropping missing values, so it was skipped.
- Recomputed Benjamini–Hochberg FDR once across all numeric + categorical p-values (23 tests).
- Built a shortlist using FDR p < 0.05, |effect| ≥ 0.10, and missingness < 60%.

**Results**
- Numeric: all 13 candidates remain significant after joint FDR. Strongest |rank-biserial|: **P_2** 0.834.
- Categorical: 10 features tested. Largest Cramér's V: **B_38** 0.528, **B_30** 0.407, **D_120** 0.241.
- Shortlist: **20** features (13 numeric + 7 categorical). Categoricals that did not meet V ≥ 0.10: D_63 (0.089), D_116 (0.056), D_126 (0.055).

Outputs: `statistical_test_results.csv`, `statistical_categorical_results.csv`, `statistical_family_summary.csv`, `statistical_shortlist.csv`.
